# Conditional VAE training

This notebook downloads the published wing dataset when needed and trains the conditional wing VAE.

Run JupyterLab from the repository root so relative paths resolve correctly.

In [ ]:
from pathlib import Path
import shutil

from huggingface_hub import snapshot_download

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Launch JupyterLab from the repository root.')

DATASET_REPOSITORY = 'jpfenniger/Aero-Struct-Wing'
WING_DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'wings'
REQUIRED_FILES = ('combined_wings_data_clipped.csv', 'config.yaml')

In [ ]:
missing_files = [name for name in REQUIRED_FILES if not (WING_DATA_DIR / name).is_file()]
if missing_files:
    WING_DATA_DIR.mkdir(parents=True, exist_ok=True)
    snapshot = Path(snapshot_download(repo_id=DATASET_REPOSITORY, repo_type='dataset'))
    for file_name in missing_files:
        matches = [path for path in snapshot.rglob(file_name) if path.is_file()]
        if not matches:
            raise FileNotFoundError(f'Missing {file_name} in {DATASET_REPOSITORY}.')
        shutil.copy2(matches[0], WING_DATA_DIR / file_name)
print(f'Wing dataset ready at {WING_DATA_DIR}')

In [ ]:
from generative_inverse_wing_design.models.utils.training_utils import train_cvae_with_hydra

model, trainer, cfg, wing_cfg, data_module = train_cvae_with_hydra()